In [21]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
path = os.path.join( "..", "data", "processed", "accidents_clean.csv")
accidents_clean = pd.read_csv(path)

In [23]:
# GLM avec distribution de Poisson pour tester l'effet de l'age sur la gravité de l'accident
# Important : Install statsmodels !


import statsmodels.api as sm
import statsmodels.formula.api as smf

glm_model = smf.glm(formula='grav ~ age',
                    data=accidents_clean,
                    family=sm.families.Poisson()).fit()

# Afficher le résumé du modèle
print(glm_model.summary())

coef = glm_model.params['age']
effet = np.exp(coef)
print(f"Effet multiplicatif de l'âge : {effet:.3f}")

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   grav   No. Observations:               618606
Model:                            GLM   Df Residuals:                   618604
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -8.5774e+05
Date:                Fri, 20 Jun 2025   Deviance:                   2.1165e+05
Time:                        16:03:41   Pearson chi2:                 2.20e+05
No. Iterations:                     4   Pseudo R-squ. (CS):          5.966e-05
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.5684      0.002    262.885      0.0

In [ ]:
# Anova pour tester l'effet de l'heure sur la gravité de l'accident
# Important : Install statsmodels !

# Convertir la colonne 'hrmn' en type datetime
accidents_clean['hrmn'] = pd.to_datetime(accidents_clean['hrmn'], format='%H:%M')

# Extraire les heures et les minutes
hours = accidents_clean['hrmn'].dt.hour
minutes = accidents_clean['hrmn'].dt.minute

# Convertir en nombre décimal
accidents_clean['hrmn_continuous'] = hours + minutes / 60

import statsmodels.api as sm
import statsmodels.formula.api as smf

result = smf.ols('grav ~ hrmn_continuous', data= accidents_clean).fit()

sm.stats.anova_lm(result)

,df,sum_sq,mean_sq,F,PR(>F)
hrmn_continuous,1.0,370.365636,370.365636,582.05633,1.537482e-128
Residual,614557.0,391045.990281,0.636305,NaN,NaN


In [ ]:
# GLM avec distribution de Poisson pour tester l'effet de l'age et la gravité sur le nombre d'accidents
# Important : Install statsmodels !

import statsmodels.api as sm
import statsmodels.formula.api as smf

accidents_age = accidents_clean.groupby(['age', 'grav']).size().reset_index(name='accident_count')

# Ajustement du GLM avec une distribution de Poisson
# La formule spécifie que 'accident_count' est modélisé en fonction de 'age' et 'grav'
glm_model = smf.glm(formula='accident_count ~ age * grav',
                    data=accidents_age,
                    family=sm.families.Poisson()).fit()

# Afficher le résumé du modèle
print(glm_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         accident_count   No. Observations:                  411
Model:                            GLM   Df Residuals:                      407
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -2.1876e+05
Date:                Tue, 17 Jun 2025   Deviance:                   4.3420e+05
Time:                        15:05:39   Pearson chi2:                 4.04e+05
No. Iterations:                     6   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      9.3282      0.005   2032.077      0.0

In [36]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from itertools import combinations

def cramers_v_corrected(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.shape[0] < 2 or confusion_matrix.shape[1] < 2:
        return np.nan, np.nan  # Trop peu de modalités

    chi2, p, _, _ = chi2_contingency(confusion_matrix, correction=False)
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape

    # Correction de biais (Bergsma)
    phi2corr = max(0, phi2 - ((k - 1)*(r - 1)) / (n - 1))
    rcorr = r - ((r - 1)**2) / (n - 1)
    kcorr = k - ((k - 1)**2) / (n - 1)

    v = np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))
    return v, p

def analyser_categorielle_v_cramer(df, cat_vars, target, seuil_redondance=0.4):
    print("\n=== 🔍 Association avec la target ===")
    cramer_target = {}
    for var in cat_vars:
        v, p = cramers_v_corrected(df[var], df[target])
        cramer_target[var] = v
        if not np.isnan(v):
            print(f"📌 {var} vs {target} → V de Cramér = {v:.3f}, p = {p:.3g}")

    print("\n=== 🔗 Détection de redondances entre variables catégorielles ===")
    redondantes = []
    for var1, var2 in combinations(cat_vars, 2):
        v, _ = cramers_v_corrected(df[var1], df[var2])
        if not np.isnan(v) and v > seuil_redondance:
            redondantes.append((var1, var2, v))
            print(f"⚠️  {var1} et {var2} sont corrélées (V = {v:.3f})")

    if not redondantes:
        print("✅ Aucune redondance forte détectée.")
    return cramer_target, redondantes

In [37]:

variables_exclues = ['hrmn','lat', 'long', 'age', 'date']
cat_vars = [col for col in accidents_clean.columns if col not in variables_exclues]
target = 'grav'

cramer_results, redondance = analyser_categorielle_v_cramer(accidents_clean, cat_vars, target)



=== 🔍 Association avec la target ===
📌 obs vs grav → V de Cramér = 0.159, p = 0
📌 obsm vs grav → V de Cramér = 0.144, p = 0
📌 choc vs grav → V de Cramér = 0.122, p = 0
📌 manv vs grav → V de Cramér = 0.162, p = 0
📌 motor vs grav → V de Cramér = 0.101, p = 0
📌 place vs grav → V de Cramér = 0.146, p = 0
📌 catu vs grav → V de Cramér = 0.175, p = 0
📌 grav vs grav → V de Cramér = 1.000, p = 0
📌 sexe vs grav → V de Cramér = 0.092, p = 0
📌 trajet vs grav → V de Cramér = 0.109, p = 0
📌 jour vs grav → V de Cramér = 0.002, p = 0.304
📌 mois vs grav → V de Cramér = 0.020, p = 8.01e-146
📌 an vs grav → V de Cramér = 0.009, p = 8.11e-27
📌 lum vs grav → V de Cramér = 0.068, p = 0
📌 dep vs grav → V de Cramér = 0.161, p = 0
📌 com vs grav → V de Cramér = 0.287, p = 0
📌 agg vs grav → V de Cramér = 0.168, p = 0
📌 int vs grav → V de Cramér = 0.054, p = 0
📌 atm vs grav → V de Cramér = 0.032, p = 0
📌 col vs grav → V de Cramér = 0.152, p = 0
📌 catr vs grav → V de Cramér = 0.108, p = 0
📌 circ vs grav → V de Cra